In [4]:
import openmc.deplete
import openmc.data
import os
from collections import defaultdict
from openmc.deplete.nuclide import ReactionTuple

# 1. Extract All Non-Fission Reactions
Get all reaction pathways OpenMC supports, but filter out 'fission'. 
Since this chain lacks fission product yields (fpy_files=[]), including fission might only bloat the matrix with placeholder yields for actinides that are not present in the present tokamak model 

In [5]:
fusion_rx = tuple(r for r in openmc.deplete.chain.REACTIONS if r != 'fission')
print("Tracking fusion reactions:", fusion_rx)


Tracking fusion reactions: ('(n,2nd)', '(n,2n)', '(n,3n)', '(n,na)', '(n,n3a)', '(n,2na)', '(n,3na)', '(n,np)', '(n,n2a)', '(n,2n2a)', '(n,nd)', '(n,nt)', '(n,n3He)', '(n,nd2a)', '(n,nt2a)', '(n,4n)', '(n,2np)', '(n,3np)', '(n,n2p)', '(n,npa)', '(n,gamma)', '(n,p)', '(n,d)', '(n,t)', '(n,3He)', '(n,a)', '(n,2a)', '(n,3a)', '(n,2p)', '(n,pa)', '(n,t2a)', '(n,d2a)', '(n,pd)', '(n,pt)', '(n,da)', '(n,5n)', '(n,6n)', '(n,2nt)', '(n,ta)', '(n,4np)', '(n,3nd)', '(n,nda)', '(n,2npa)', '(n,7n)', '(n,8n)', '(n,5np)', '(n,6np)', '(n,7np)', '(n,4na)', '(n,5na)', '(n,6na)', '(n,7na)', '(n,4nd)', '(n,5nd)', '(n,6nd)', '(n,3nt)', '(n,4nt)', '(n,5nt)', '(n,6nt)', '(n,2n3He)', '(n,3n3He)', '(n,4n3He)', '(n,3n2p)', '(n,3n2a)', '(n,3npa)', '(n,dt)', '(n,npd)', '(n,npt)', '(n,ndt)', '(n,np3He)', '(n,nd3He)', '(n,nt3He)', '(n,nta)', '(n,2n2p)', '(n,p3He)', '(n,d3He)', '(n,3Hea)', '(n,4n2p)', '(n,4n2a)', '(n,4npa)', '(n,3p)', '(n,n3p)', '(n,3n2pa)', '(n,5n2p)')



# 2. Load the Raw Nuclear Data (Decay and Activation)
Pulls in the JEFF-3.3 decay evaluations and collects the FENDL-3.0 neutron activation ENDF files. 
Uses a try/except block to safely filter out any documentation or non-ENDF files in the directory that would otherwise crash the parser.

In [7]:
print("Unpacking....")
decay_evals = openmc.data.endf.get_evaluations("/workspace/JEFF33-rdd_all.asc")   #must copy the JEFF33-rdd_all.asc in the docker container.

raw_files = []
for root, dirs, files in os.walk("/workspace/endf"):
    for file in files:
        if not file.startswith('.'):
            raw_files.append(os.path.join(root, file))

neutron_files = []
for f in raw_files:
    try:
        # Test-parse to ensure the file is a valid ENDF evaluation
        openmc.data.endf.Evaluation(f)
        neutron_files.append(f)
    except Exception:
        pass
print("JEFF-3.3 unpacked")

Unpacking....
JEFF-3.3 unpacked


# 3. Constructing Initial Base Chain
We feed the raw data and our expanded fusion reaction list into OpenMC's parser.
'''

In [8]:
print("Building base fusion chain...")
fusion_chain = openmc.deplete.Chain.from_endf(
    decay_files=decay_evals,
    fpy_files=[],
    neutron_files=neutron_files,
    reactions=fusion_rx
)
print("done")

Building base fusion chain...
Processing neutron sub-library files...
Processing decay sub-library files...
Processing fission product yield sub-library files...
Creating depletion_chain...
The following fissionable nuclides have no fission product yields:
  Rn211, replaced with U235
  Rn222, replaced with U235
  Ra223, replaced with U235
  Ra224, replaced with U235
  Ra225, replaced with U235
  Ra226, replaced with U235
  Ra228, replaced with U235
  Ac225, replaced with U235
  Ac226, replaced with U235
  Ac227, replaced with U235
  Ac228, replaced with U235
  Th227, replaced with U235
  Th228, replaced with U235
  Th229, replaced with U235
  Th230, replaced with U235
  Th231, replaced with U235
  Th232, replaced with U235
  Th234, replaced with U235
  Pa228, replaced with U235
  Pa229, replaced with U235
  Pa230, replaced with U235
  Pa231, replaced with U235
  Pa232, replaced with U235
  Pa233, replaced with U235
  Pa234, replaced with U235
  U230, replaced with U235
  U231, replaced

# 4. Patch Missing (n,2n) and Metastable Channels
Load a standard fission chain and copy over any missing reaction channels, ensuring the target isotope actually exists in the new fusion chain to avoid missing pointers.

In [9]:

print("Patching...")
old_chain = openmc.deplete.Chain.from_xml('/nuclear_data/endfb-viii.0-hdf5/chain_endfb80_sfr.xml')

for nuc_name, old_idx in old_chain.nuclide_dict.items():
    if nuc_name in fusion_chain.nuclide_dict:
        old_nuc = old_chain.nuclides[old_idx]
        new_nuc = fusion_chain.nuclides[fusion_chain.nuclide_dict[nuc_name]]
        
        existing_rx = {(r.type, r.target) for r in new_nuc.reactions}
        
        for r in old_nuc.reactions:
            if (r.type, r.target) not in existing_rx and r.target in fusion_chain.nuclide_dict:
                new_nuc.reactions.append(r)

print("Missing cross-sections from old chain patched")


Patching...
Missing cross-sections from old chain patched


# 5. Normalize Branching Ratios
The patching process can artificially inflate or deflate branching ratios (e.g., ground + metastable states not equal to 1). 
This calculates the total for each reaction type and normalizes them back to exactly 1.0 to prevent material loss or creation.

In [10]:
print("Normalizing branching ratios...")
for nuc in fusion_chain.nuclides:
    by_type = defaultdict(list)
    for i, r in enumerate(nuc.reactions):
        by_type[r.type].append(i)
        
    for typ, idxs in by_type.items():
        tot = sum(nuc.reactions[i].branching_ratio for i in idxs)
        # Check for both > 1 and < 1 deviations
        if abs(tot - 1.0) > 1e-6 and tot > 0:
            for i in idxs:
                r = nuc.reactions[i]
                nuc.reactions[i] = ReactionTuple(r.type, r.target, r.Q, r.branching_ratio / tot)


Normalizing branching ratios...


# 6. Explicitly binding the Tritium Breeding Reactions
Using ReactionTuple ensures no IndexError if an isotope is initialized with zero reactions.

In [11]:
li6_nuc = fusion_chain.nuclides[fusion_chain.nuclide_dict['Li6']]
if not any(r.type == '(n,t)' for r in li6_nuc.reactions):
    li6_nuc.reactions.append(ReactionTuple('(n,t)', 'H3', 4.783e6, 1.0))
    print("Forced Li-6 (n,t) Tritium breeding channel.")

li7_nuc = fusion_chain.nuclides[fusion_chain.nuclide_dict['Li7']]
if not any(r.target == 'H3' for r in li7_nuc.reactions):
    # Li-7(n,n't)a reaction producing Tritium
    li7_nuc.reactions.append(ReactionTuple('(n,na)', 'H3', -2.467e6, 1.0))
    print("Forced Li-7 Tritium breeding channel.")


# 7. Validation, Library Checks, and Provenance Printouts
Verify MT=22 (n,na) actually exists in the Li-7 transport library. Print any dropped reaction names and output specific nuclide channels to verify the patches.

In [12]:
print("Checking Li-7 transport library for MT=22...")
try:
    li7_transport = openmc.data.IncidentNeutron.from_hdf5('/nuclear_data/fendl-3.2-hdf5/neutron/Li7.h5')
    print("MT=22 in Li-7 transport library:", 22 in li7_transport.reactions)
except Exception as e:
    print("Could not load Li7.h5 library for check:", e)

print('Dropped reaction names:', set(openmc.deplete.chain.REACTIONS) - set(fusion_chain.reactions))

print("Validating chain branching sums...")
fusion_chain.validate(strict=False, quiet=False)

print("Key isotope reaction check:")
for nuc in ['Li6', 'Li7', 'W184', 'W186', 'Pb204', 'Cu63']:
    if nuc in fusion_chain.nuclide_dict:
        n = fusion_chain.nuclides[fusion_chain.nuclide_dict[nuc]]
        print(nuc, [(r.type, r.target, round(r.branching_ratio, 3)) for r in n.reactions])


Checking Li-7 transport library for MT=22...
MT=22 in Li-7 transport library: False
Dropped reaction names: set()
Validating chain branching sums...
Key isotope reaction check:
Li6 [('(n,2na)', 'H1', 1.0), ('(n,nd)', 'He4', 1.0), ('(n,gamma)', 'Li7', 1.0), ('(n,p)', 'He6', 1.0), ('(n,t)', 'He4', 1.0)]
Li7 [('(n,2n)', 'Li6', 1.0), ('(n,na)', 'H3', 1.0), ('(n,2na)', 'H2', 1.0), ('(n,np)', 'He6', 1.0), ('(n,gamma)', 'Li8', 1.0), ('(n,d)', 'He6', 1.0)]
W184 [('(n,2nd)', 'Ta181', 1.0), ('(n,3n)', 'W182', 1.0), ('(n,np)', 'Ta183', 1.0), ('(n,2n2a)', 'Yb175', 1.0), ('(n,nt)', 'Ta181', 1.0), ('(n,n3He)', 'Hf181', 1.0), ('(n,4n)', 'W181', 1.0), ('(n,3np)', 'Ta181', 1.0), ('(n,npa)', 'Lu179', 1.0), ('(n,gamma)', 'W185', 0.895), ('(n,p)', 'Ta184', 1.0), ('(n,d)', 'Ta183', 1.0), ('(n,a)', 'Hf181', 1.0), ('(n,2p)', 'Hf183', 1.0), ('(n,5n)', 'W180', 1.0), ('(n,7n)', 'W178', 1.0), ('(n,8n)', 'W177', 1.0), ('(n,5np)', 'Ta179', 1.0), ('(n,7np)', 'Ta177', 1.0), ('(n,5na)', 'Hf176', 1.0), ('(n,6na)', 'Hf

# 8. Export the Final Chain

In [13]:
output_path = "/workspace/chain/chain_fendl30_fusion_complete.xml"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
fusion_chain.export_to_xml(output_path)
print(f"Success! Complete fusion chain saved to: {output_path}")

Success! Complete fusion chain saved to: /workspace/chain/chain_fendl30_fusion_complete.xml
